In [1]:
import os
import re
import json
import random
os.chdir(os.path.dirname(os.getcwd()))

In [2]:
from helpers.utils import walk_dir

In [3]:
evaluation_fps = walk_dir("data/models",keyword='evaluation',extension='json')

In [45]:
d.keys()

dict_keys(['question', 'category', 'agristack_required', 'farmer_id', 'answer', 'agent_turns', 'evaluation'])

In [43]:
evaluation_fps

['data/models/gpt-4.1-mini/evaluation.json',
 'data/models/gpt-4.1/evaluation.json',
 'data/models/mistralai_Mistral-Small-3.2-24B-Instruct-2506/evaluation.json',
 'data/models/openai_gpt-oss-20b/evaluation.json']

In [9]:
for eval_fp in evaluation_fps:
    model_name  = os.path.split(eval_fp)[0].split("/")[-1]
    data = json.load(open(eval_fp,'r'))
    break

In [17]:
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field

class Metric(BaseModel):
    score: int = Field(..., ge=0, le=5, description="Higher is better")
    explanation: str
    evidence: Optional[List[str]] = Field(None, description="Optional short snippets/pointers showing why this score was given")

class ProtocolCompliance(BaseModel):
    scope_discipline: Metric
    mandatory_tool_compliance: Metric
    agristack_workflow_compliance: Metric
    term_identification_compliance: Metric
    query_construction_discipline: Metric
    tool_selection_and_sequencing_quality: Metric

class GroundingAndTruthfulness(BaseModel):
    groundedness: Metric
    citation_claim_alignment: Metric
    factual_correctness_given_sources: Metric
    non_fabrication_discipline: Metric  # high = no invention / good uncertainty handling

class SafetyPolicy(BaseModel):
    safety_and_regulatory_compliance: Metric

class ResponseQuality(BaseModel):
    actionability: Metric
    completeness: Metric
    context_fit: Metric
    clarity_and_readability: Metric
    conversation_closure_quality: Metric  # high = ends with relevant follow-up

class MarathiQuality(BaseModel):
    marathi_linguistic_quality: Metric
    marathi_terminology_fidelity: Metric

class IntegrityHygiene(BaseModel):
    tool_name_confidentiality: Metric  # high = no leakage

class EvaluationResult(BaseModel):
    protocol: ProtocolCompliance
    grounding: GroundingAndTruthfulness
    safety: SafetyPolicy
    response: ResponseQuality
    integrity: IntegrityHygiene
    marathi: Optional[MarathiQuality] = None  # include only if selected_language == "mr"

    overall_score: int = Field(..., ge=0, le=100)
    summary: str = Field(..., description="1–3 lines: strongest points + top improvement")


In [30]:
def create_label(key):
    """Create a label from a key"""
    return re.sub("_"," ",key).title()

In [90]:
d = random.choice(data)
eval_result = EvaluationResult.model_validate(d['evaluation'])

In [91]:
keys = ['protocol', 'grounding', 'safety', 'response', 'integrity', 'marathi', 'overall_score', 'summary']

In [95]:
metric_data

{'score': 0,
 'explanation': 'No farmer‑friendly citations provided in the final answer.',
 'evidence': None}

In [100]:
import pandas as pd

In [102]:
df = pd.read_csv("/Users/adityachhabra/Github/openagrinet/oan-evaluation/data/evaluation_questions.csv")

In [104]:
df.category.unique()

array(['MahaDBT', 'Warehouse', 'CHC', 'KVK', 'Soil Labs', 'Advisory',
       'Weather Historical', 'Agri Assistant Contact', 'Mandi', 'Scheme',
       'Weather Forecast'], dtype=object)

In [107]:
df.category.value_counts()

category
Advisory                  11
MahaDBT                   10
Mandi                     10
Warehouse                  9
CHC                        9
KVK                        9
Soil Labs                  9
Agri Assistant Contact     9
Scheme                     9
Weather Forecast           9
Weather Historical         8
Name: count, dtype: int64

In [ ]:
MahaDBT मदत मिळण्यासाठी ई-केवायसी अनिवार्य आहे

In [99]:
for key, metric_data in eval_result.grounding.model_dump().items():
    label = create_label(key)
    score, explanation = metric_data['score'], metric_data['explanation']
    #, '\n'.join([e for e in metric_data['evidence']])
    print(score)
        

2
0
2
2
